# Phase 4 - Experiment 4: A (plain decoder) + bottleneck adapters

**Model:** Phase-1 A backbone (10 causal layers, 70M), **frozen**. After the attention and after the feed-forward of every block we insert a small **adapter** (Houlsby et al. 2019):

`out = y + W_up( GELU( W_down(y) ) )`, y = output of the sub-layer, W_down 640 -> 64, W_up 64 -> 640 (starts at 0).

20 adapters x ~83k = ~1.65M new parameters (+ head), lr 5e-4. Difference to LoRA/LoRMA: those modify the existing weight matrices with a low-rank product; an adapter is a NEW layer with a non-linearity.

**Decisions (from P4_01), shared by all Phase-4 experiments:**
| decision | why |
|---|---|
| remove exact duplicate texts -> 31,652 train / 994 dev cases | ILDC_single is inside ILDC_multi; dev lists every case twice |
| drop the labels in test.json | they must never be used |
| lower-case every split, repair `peti-\ntion` line wraps | train is cased, dev/test are lower-case old cases |
| read the **last 1536 tokens** of each case | the end carries most signal (Naive Bayes: last 512 > first 512); whole case would cost ~4 h/epoch |
| class-weighted loss (w0 = 0.85, w1 = 1.21) + decision threshold tuned on dev | train is 41% label 1, dev/test are 50% |
| no Stage B (retrain on train + dev) | dev is 3% of train; a second run would cost hours |

**Time:** ~31.6k cases x ~1,480 tokens ≈ 47M tokens per epoch -> roughly 35-60 min per epoch on the RTX 4050; 3 epochs.
Checkpoint every 250 steps: if anything stops, just run the notebook again and it continues.
Everything is cached in `checkpoints/phase4/A_adapter64_causal_last1536_attn/` (if `scripts/p4_run.py` already ran it, every cell just loads).

In [ ]:
import sys, json
from pathlib import Path
import torch

ROOT = Path.cwd().resolve()
while not (ROOT / ".gitignore").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Project root with .gitignore not found")
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
import importlib, cjpe_utils, cjpe_models, cjpe_experiment
for m in (cjpe_utils, cjpe_models, cjpe_experiment):
    importlib.reload(m)
from cjpe_experiment import Experiment, compare

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU would be far too slow)")

## 1. Data (first run: cleaning + tokenising ~31.6k cases takes a few minutes, then cached in `data/processed/phase4/`)

In [ ]:
CONFIG_DICT = {"phase1": "A", "method": "adapter"}
exp = Experiment(ROOT, CONFIG_DICT)
print(json.dumps({k: v for k, v in exp.cfg.items() if k != "adapter_targets"}))
import numpy as np
n = np.array([len(x) for x in exp.X["train"]])
print(f"tokens the model reads per training case: mean {n.mean():.0f}, {100 * (n == exp.cfg['max_len']).mean():.0f}% cut to {exp.cfg['max_len']}")
print("example (decoded end of the first dev case):")
from tokenizers import Tokenizer
tok = Tokenizer.from_file(str(ROOT / "data" / "tokenizer" / "bpe32k.json"))
print("  ...", tok.decode(exp.X["dev"][0][-80:].tolist()))

## 2. Memory check (one forward + backward on a full micro-batch of 2 x 1536 tokens)

In [ ]:
exp.memory_check()

## 3. Train (long: ~3 epochs x ~50 min). Re-running this cell resumes / loads the finished model.

In [ ]:
state = exp.train()
for row in state["history"]:
    print(row)

## 4. Dev report

- macro-F1 at threshold 0.5 and at the threshold tuned on dev (the model saw 41% label 1, dev has 50%)
- confusion matrix, score by case length, the most confident false positives / negatives (ids, for the error analysis)

In [ ]:
rep = exp.report()

## 5. Test predictions -> submission.zip (dev-tuned threshold)

In [ ]:
zip_path = exp.predict_test()

## 7. Speed, log, comparison table

In [ ]:
exp.speed()
exp.log()
compare(ROOT)